In [0]:
lms      = spark.table("edtech_project.edtech_bronze.lms_events_bronze") 
sessions = spark.table("edtech_project.edtech_silver.learning_sessions")
quiz_df  = spark.read.table("edtech_project.edtech_bronze.quiz_attempts_bronze")
video_df = spark.read.table("edtech_project.edtech_bronze.video_events_bronze") 
discussion = spark.read.table("edtech_project.edtech_silver.discussion_posts_parsed")



In [0]:
display(lms)

In [0]:
video_dff = spark.read.table("edtech_project.edtech_bronze.video_events_bronze") 

In [0]:
video_dff.columns

In [0]:
display(video_dff)

In [0]:
from pyspark.sql.functions import col

video_dff = video_dff.withColumn(
    "video_minutes_watched",
    col("watched_seconds") / 60
)

video_dff.show()


In [0]:
from pyspark.sql.functions import sum, avg, round

video_features = video_dff.groupBy("student_id", "course_id").agg(
    round(sum("video_minutes_watched"),2).alias("total_video_minutes_watched"),
    round(avg("completion_pct"),2).alias("video_completion_rate")
)

In [0]:
display(video_features)

In [0]:
from pyspark.sql.functions import col, coalesce, lit

lms_clean = lms.withColumn(
    "duration_seconds",
    coalesce(col("duration_seconds").cast("double"), lit(0.0))
)


In [0]:
quiz_df.printSchema()

In [0]:
from pyspark.sql.functions import col

quiz_clean = quiz_df \
    .withColumn("score_obtained", col("score_obtained").cast("double")) \
    .withColumn("max_score", col("max_score").cast("double")) \
    .withColumn("pass_threshold", col("pass_threshold").cast("double"))

In [0]:
quiz_df.printSchema()

In [0]:
# quizz = quiz_clean.groupBy("student_id","course_id") 


In [0]:
from pyspark.sql.functions import max, when, col

quiz_level = quiz_df.groupBy(
    "student_id", "course_id", "quiz_id"
).agg(
    max(when(col("status") == "PASSED", 1).otherwise(0)).alias("quiz_passed"),
    max("attempt_number").alias("attempts_per_quiz")
)


In [0]:
display(quiz_level)

In [0]:
from pyspark.sql.functions import avg, sum,round

quiz_features = quiz_level.groupBy(
    "student_id","course_id","quiz_id"
).agg(
    sum("attempts_per_quiz").alias("quiz_attempts_count"),
    round((avg("quiz_passed")*100)/col("quiz_attempts_count"), 4).alias("quiz_pass_rate")
)

In [0]:
display(quiz_features)
# quiz_features.printSchema()

In [0]:
from pyspark.sql.functions import current_date, datediff

login_df = lms_clean.filter(col("event_type") == "login")

login_features = login_df.groupBy("student_id", "course_id").agg(
    sum(when(datediff(current_date(), col("event_ts")) <= 7, 1).otherwise(0)).alias("login_count_7d"),
    sum(when(datediff(current_date(), col("event_ts")) <= 30, 1).otherwise(0)).alias("login_count_30d")
)

In [0]:
display(login_features)

In [0]:
from pyspark.sql import functions as F 

last_df = spark.read.table("edtech_project.edtech_bronze.enrollments_bronze")
# display(last_df)

relevant_status_df = last_df.filter(last_df.status.isin(['ACTIVE', 'PAUSED', 'DROPPED']))
# display(relevant_status_df)

last_active_df = relevant_status_df.groupBy('student_id', 'course_id') \
    .agg(F.max('last_accessed_date').alias('last_active_date'))

last_active_df = last_active_df.dropna(subset=['last_active_date'])

# display(last_active_df)
display(last_df)

In [0]:
# according to data we can change this
last_active_df = last_active_df.withColumn(
    'days_since_last_active',
    F.datediff(F.current_date(), F.col('last_active_date'))
)

In [0]:
display(last_active_df)

In [0]:
from pyspark.sql.functions import weekofyear

weeklyy = lms_clean.withColumn("week", weekofyear(col("event_date"))) 

In [0]:
display(weeklyy)

In [0]:
weekly_counts = weeklyy.groupBy("student_id", "course_id", "week").count()

In [0]:
display(weekly_counts)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("student_id", "course_id").orderBy("week")

weekly_counts = weekly_counts.withColumn(
    "prev_week_count",
    lag("count").over(window_spec)
)

In [0]:
weekly_counts = weekly_counts.fillna(value=0, subset=['prev_week_count'])

In [0]:
display(weekly_counts)

In [0]:
from pyspark.sql.functions import when

weekly_counts = weekly_counts.withColumn(
    "engagement_trend",
    when(col("count") > col("prev_week_count"), "increasing")
    .when(col("count") < col("prev_week_count"), "decreasing")
    .otherwise("stable")
)

In [0]:
display(weekly_counts)

In [0]:
from pyspark.sql.functions import sum

discussion_posts_per_course_per_student = discussion.groupBy("author_student_id", "course_id").agg(
    sum("student_posts").alias("total_posts")
)

In [0]:
discussion_posts_per_course_per_student = discussion_posts_per_course_per_student.withColumnRenamed("author_student_id", "student_id")

In [0]:
display(discussion_posts_per_course_per_student)

In [0]:
# engagement_features = lms_clean.groupBy("student_id", "course_id").agg(
#     sum(when(col("event_type") == "discussion_post", 1).otherwise(0)).alias("discussion_posts_count"),
#     sum(when(col("event_type") == "assignment_submit", 1).otherwise(0)).alias("assignment_submissions")
# )

In [0]:
# final_df = quiz_features \
#     .join(video_features, ["student_id", "course_id"], "left") 

In [0]:
# display(final_df)

In [0]:
# display(final_df)

In [0]:
from pyspark.sql.functions import col, trim, upper

def clean_keys(df):
    return df.withColumn("student_id", trim(upper(col("student_id")))) \
             .withColumn("course_id", trim(upper(col("course_id"))))

quiz_features = clean_keys(quiz_features)
video_features = clean_keys(video_features)
discussion_posts_per_course_per_student = clean_keys(discussion_posts_per_course_per_student)
last_active_df = clean_keys(last_active_df)

In [0]:
display(quiz_features)

In [0]:
display(video_features)

In [0]:
base_df = quiz_features.select("student_id", "course_id") \
    .union(video_features.select("student_id", "course_id")) \
    .union(discussion_posts_per_course_per_student.select("student_id", "course_id")) \
    .union(last_active_df.select("student_id", "course_id")) \
    .distinct()

In [0]:
base_df.count()

In [0]:
quiz_sel = quiz_features.select(
    "student_id",
    "course_id",
    "quiz_pass_rate",
    "quiz_attempts_count"
)

video_sel = video_features.select(
    "student_id",
    "course_id",
    "total_video_minutes_watched",
    "video_completion_rate"
)

posts_sel = discussion_posts_per_course_per_student.select(
    "student_id",
    "course_id",
    "total_posts"
)

active_sel = last_active_df.select(
    "student_id",
    "course_id",
    "last_active_date",
    "days_since_last_active"
)

In [0]:
final_df = base_df \
    .join(quiz_sel, ["student_id", "course_id"], "left") \
    .join(video_sel, ["student_id", "course_id"], "left") \
    .join(posts_sel, ["student_id", "course_id"], "left")  \
    .join(active_sel, ["student_id", "course_id"], "left")

In [0]:
final_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("edtech_project.edtech_silver.student_course_engagement")

In [0]:
display(final_df)

In [0]:
videos = spark.table('edtech_project.edtech_bronze.video_events_bronze')
videos = videos.filter(F.col('student_id') == 'STU000594')
display(videos)

In [0]:
lms = spark.table('edtech_project.edtech_bronze.lms_events_bronze')
# lms = lms.filter(F.col('student_id') == 'STU000594')
lms.count()

In [0]:
from pyspark.sql import functions as F
quiz = spark.table('edtech_project.edtech_bronze.quiz_attempts_bronze')

quiz = quiz.filter((F.col('student_id') == 'STU000524') & (F.col('course_id') == 'CRS0111'))
display(quiz)

In [0]:
from pyspark.sql import functions as F
en = spark.table('edtech_project.edtech_bronze.enrollments_bronze')

en = en.filter((F.col('student_id') == 'STU000524') & (F.col('course_id') == 'CRS0111'))
display(en)

In [0]:
from pyspark.sql import functions as F
en = spark.table('edtech_project.edtech_bronze.enrollments_bronze')

# en = en.filter((F.col('student_id') == 'STU000524') & (F.col('course_id') == 'CRS0111'))
display(en)

In [0]:
joined = quiz.join(en, ['student_id', 'course_id'], 'inner')
display(joined)

In [0]:
ab=last_active_df.join(video_features, ['student_id', 'course_id'], 'inner')
display(ab)

In [0]:
display(final_df)